# Accuracy test for the KD project (Role 4)

Tests the 3 saved models (teacher, baseline student, distilled student) on the 10,000 CIFAR-10 test pictures.
It uses the SAME student definition and normalisation as the team's notebooks, and compares your results with the team's `benchmark_summary.csv` that it reads from the checkpoints folder.

**How to run:** Colab > Runtime > Change runtime type > GPU, then Runtime > Run all.
Needs the `checkpoints` folder visible in My Drive (Drive > right-click checkpoints > Organize > Add shortcut > My Drive).
The first run downloads CIFAR-10 (about 170 MB, it can be slow).

In [ ]:
# 1. Connect Drive and find the model files
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT = '/content/drive/MyDrive/checkpoints'
import datetime
for f in sorted(os.listdir(CKPT)):
    t = datetime.datetime.fromtimestamp(os.path.getmtime(os.path.join(CKPT, f)))
    print(f'{f:28s} last modified {t:%d %b %H:%M}')

In [ ]:
# 2. Imports and model definitions (identical to Role 3's notebook)
import time, tempfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader
from torchvision.models import resnet18
from sklearn.metrics import confusion_matrix, f1_score, ConfusionMatrixDisplay

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
CLASSES = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
COLORS = {'Teacher': '#1f77b4', 'Baseline': '#7f7f7f', 'KD': '#2ca02c'}
os.makedirs('charts', exist_ok=True); os.makedirs('results', exist_ok=True)

class StudentCNN(nn.Module):
    def __init__(self, num_classes=10, dropout_rate=0.2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 4 * 4, 128), nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate), nn.Linear(128, num_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

def make_teacher():
    m = resnet18()
    m.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    m.maxpool = nn.Identity()
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m

In [ ]:
# 3. CIFAR-10 test set (same normalisation as Role 2 and Role 3: std 0.2470, 0.2435, 0.2616)
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
test_set = torchvision.datasets.CIFAR10('./data', train=False, download=True,
                                        transform=T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)]))
loader = DataLoader(test_set, batch_size=256, shuffle=False)
print('Test pictures:', len(test_set))

In [ ]:
# 4. Load the three real models
def load(model, fname):
    model.load_state_dict(torch.load(os.path.join(CKPT, fname), map_location='cpu'))
    return model.to(device).eval()

models = {
    'Teacher':  load(make_teacher(), 'teacher_model.pth'),
    'Baseline': load(StudentCNN(), 'student_baseline.pt'),
    'KD':       load(StudentCNN(), 'student_kd.pt'),
}
print('All 3 models loaded.')

In [ ]:
# 5. THE ACCURACY TEST
@torch.no_grad()
def predict(model):
    ys, ps, loss_sum = [], [], 0.0
    ce = nn.CrossEntropyLoss(reduction='sum')
    for x, y in loader:
        out = model(x.to(device))
        loss_sum += ce(out, y.to(device)).item()
        ps.append(out.argmax(1).cpu()); ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy(), loss_sum / len(loader.dataset)

def size_mb(model):
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f: p = f.name
    torch.save(model.state_dict(), p); s = os.path.getsize(p) / 1e6; os.remove(p); return s

@torch.no_grad()
def latency_ms(model, runs=100):
    x = torch.randn(1, 3, 32, 32, device=device)
    for _ in range(20): model(x)
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs): model(x)
    if device == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs * 1000

try:
    bench = pd.read_csv(os.path.join(CKPT, 'benchmark_summary.csv'))
    acc_col = [c for c in bench.columns if 'Accuracy' in c][0]
    team = dict(zip(['Teacher', 'Baseline', 'KD'], bench[acc_col].astype(float).tolist()[:3]))
except Exception as e:
    print('Could not read benchmark_summary.csv:', e); team = {'Teacher': float('nan'), 'Baseline': float('nan'), 'KD': float('nan')}
print('Team CSV accuracies:', team)
preds, rows = {}, []
for name, m in models.items():
    y, p, loss = predict(m)
    preds[name] = p
    acc = (y == p).mean() * 100
    rows.append({'Model': name, 'Accuracy (%)': round(acc, 2), 'Team CSV (%)': team[name],
                 'Difference': round(acc - team[name], 2), 'Test loss': round(loss, 4),
                 'Macro F1': round(f1_score(y, p, average='macro'), 4),
                 'Params (learnable)': sum(q.numel() for q in m.parameters()),
                 'Size (MB)': round(size_mb(m), 2), 'Latency (ms)': round(latency_ms(m), 3)})
y_true = y
final = pd.DataFrame(rows)
final['Smaller than teacher (x)'] = (final.loc[0, 'Params (learnable)'] / final['Params (learnable)']).round(1)
final.to_csv('results/accuracy_test.csv', index=False)
display(final)
print('If every Difference is within about 0.1, your test matches the team. Latency will differ from the team, that is normal.')

In [ ]:
# 6. Charts: accuracy bars, confusion matrices, per-class accuracy
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(final['Model'], final['Accuracy (%)'], color=[COLORS[n] for n in final['Model']])
for b, v in zip(bars, final['Accuracy (%)']):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.5, f'{v:.2f}%', ha='center')
ax.set_ylim(70, 100); ax.set_ylabel('Test accuracy (%)'); ax.set_title('Test accuracy on CIFAR-10 (single run)')
plt.tight_layout(); plt.savefig('charts/t1_accuracy_bar.png', dpi=200); plt.show()

cms = {n: confusion_matrix(y_true, p) for n, p in preds.items()}
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, n in zip(axes, models):
    ConfusionMatrixDisplay(cms[n], display_labels=CLASSES).plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
    ax.set_title(n)
plt.tight_layout(); plt.savefig('charts/t2_confusion_matrices.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(10, 4)); x = np.arange(10); w = 0.27
for i, n in enumerate(models):
    ax.bar(x + (i - 1) * w, cms[n].diagonal() / cms[n].sum(1) * 100, w, color=COLORS[n], label=n)
ax.set_xticks(x, CLASSES); ax.set_ylabel('Accuracy (%)'); ax.set_title('Accuracy per class'); ax.legend()
plt.tight_layout(); plt.savefig('charts/t3_per_class.png', dpi=200); plt.show()

In [ ]:
# 7. Is distilled really different from baseline? (paired test on the same 10,000 pictures)
from scipy.stats import binomtest
base_ok = preds['Baseline'] == y_true
kd_ok = preds['KD'] == y_true
only_base = int((base_ok & ~kd_ok).sum())   # baseline right, KD wrong
only_kd = int((~base_ok & kd_ok).sum())     # KD right, baseline wrong
p_value = binomtest(only_base, only_base + only_kd, 0.5).pvalue
print(f'Baseline right and KD wrong: {only_base} pictures')
print(f'KD right and baseline wrong: {only_kd} pictures')
print(f'McNemar p-value: {p_value:.3f}')

rng = np.random.default_rng(0)
diffs = [(kd_ok[i].mean() - base_ok[i].mean()) * 100 for i in (rng.integers(0, len(y_true), len(y_true)) for _ in range(2000))]
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f'KD minus Baseline accuracy: {(kd_ok.mean() - base_ok.mean()) * 100:+.2f} points (95% interval {lo:+.2f} to {hi:+.2f})')
print('p < 0.05 or an interval that does not include 0 = a real difference. Otherwise it is within noise.')

In [ ]:
# 8. Save everything and download
import shutil
shutil.make_archive('role4_accuracy_outputs', 'zip', '.', 'charts')
shutil.copy('results/accuracy_test.csv', 'accuracy_test.csv')
from google.colab import files
files.download('accuracy_test.csv')
files.download('role4_accuracy_outputs.zip')